### Theory

Bayes Theorem: ```Posterior{P(y|X)} ∝ Likelihood {P(X|y)} * Prior {P(y)}```

If we know the ```Likelihood P(X|y) and the Prior P(y)```, we can calulate the ```Posterior P(y|X)```, the target. But how will we estimate the Likelihood from the data?

Enter the "Naive" part: we ASSUME every feature in X is conditionally independent of every other feature given the class y. $\newline$This allows us to estimate the Likelihood ```P(X|y)``` as $\prod$ ```P(x_i|y)```

We use a suitable distribution to model the probability:

Continuous - Gaussian distribution

Binary - Bernouli distribution

Counts - Multimonial distribution




### 1. Bayes' Theorem (The Foundation)

For a sample $X = (x_1, x_2, ..., x_n)$ and a class $y$, the posterior probability is:

$$
P(y \mid X) = \frac{P(y) \cdot P(X \mid y)}{P(X)}
$$

Since $P(X)$ is constant for all classes, we only need to maximize the numerator:

$$
P(y \mid X) \propto P(y) \cdot P(X \mid y)
$$

---

### 2. The "Naive" Conditional Independence Assumption

Naive Bayes assumes that all features $x_i$ are **conditionally independent** given the class $y$. This allows us to factorize the joint likelihood:

$$
P(X \mid y) = \prod_{i=1}^{n} P(x_i \mid y)
$$

Thus, the decision rule becomes:

$$
\hat{y} = \arg\max_{y} \left[ P(y) \cdot \prod_{i=1}^{n} P(x_i \mid y) \right]
$$

---

### 3. Gaussian (Normal) Probability Density Function

For continuous features, we assume each $P(x_i \mid y)$ follows a Gaussian distribution with class-specific mean $\mu_{y,i}$ and variance $\sigma_{y,i}^2$:

$$
P(x_i \mid y) = \frac{1}{\sqrt{2\pi \sigma_{y,i}^2}} \cdot \exp\left( -\frac{(x_i - \mu_{y,i})^2}{2\sigma_{y,i}^2} \right)
$$

---

### 4. The Log-Likelihood (Numerical Stability)

Multiplying many small probabilities causes numerical underflow. To prevent this, we take the **natural logarithm** ($\log$), which turns multiplication into addition:

$$
\log P(y \mid X) \propto \log P(y) + \sum_{i=1}^{n} \log P(x_i \mid y)
$$

Substituting the Gaussian PDF and simplifying:

$$
\log P(y \mid X) \propto \log P(y) + \sum_{i=1}^{n} \left[ 
    -\frac{1}{2} \log(2\pi \sigma_{y,i}^2) 
    - \frac{(x_i - \mu_{y,i})^2}{2\sigma_{y,i}^2}
\right]
$$

---

### 5. Decomposing the Log-Likelihood (The Two Key Parts)

We can split the sum into two distinct components to match the implementation:

**A. The Normalization Constant ($\text{const}_y$):**
*(Depends only on the variance of the training data for class $y$)*

$$
\text{const}_y = -\frac{1}{2} \sum_{i=1}^{n} \log(2\pi \sigma_{y,i}^2)
$$

**B. The Quadratic Distance Penalty ($\text{exp}_y(X)$):**
*(Depends on the input sample $X$)*

$$
\text{exp}_y(X) = -\frac{1}{2} \sum_{i=1}^{n} \frac{(x_i - \mu_{y,i})^2}{\sigma_{y,i}^2}
$$

Thus, the total log-likelihood for class $y$ is:

$$
\log P(y \mid X) \propto \log P(y) + \text{const}_y + \text{exp}_y(X)
$$

---

### 6. The Final Decision Rule

To classify a new sample $X$, we compute the log-posterior for every class $y$ and select the one with the highest value:

$$
\boxed{ \hat{y} = \arg\max_{y} \left[ \log P(y) - \frac{1}{2} \sum_{i=1}^{n} \log(2\pi \sigma_{y,i}^2) - \frac{1}{2} \sum_{i=1}^{n} \frac{(x_i - \mu_{y,i})^2}{\sigma_{y,i}^2} \right] }
$$

---



### Implementation

In [1]:
import numpy as np

In [ ]:
import numpy as np

class GaussianNaiveBayes:
    """
    Gaussian Naive Bayes Classifier implemented from scratch using NumPy.
    Supports multi-class classification.
    """
    def __init__(self, var_smoothing=1e-9):
        """
        var_smoothing: Added to variance to prevent division by zero and 
                       stabilize calculations (also acts as a weak regularizer).
        """
        self.var_smoothing = var_smoothing
        self.classes_ = None
        self.priors_ = None           # P(y) for each class
        self.means_ = None            # Mean of each feature per class
        self.variances_ = None        # Variance of each feature per class

    def fit(self, X, y):
        """
        Fit the Gaussian Naive Bayes model.
        X: numpy array of shape (n_samples, n_features)
        y: numpy array of shape (n_samples,)
        """
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        # 1. Calculate Priors: P(y) = count(y) / n_samples
        self.priors_ = np.zeros(n_classes)
        for idx, cls in enumerate(self.classes_):
            self.priors_[idx] = np.sum(y == cls) / n_samples

        # 2. Calculate Means: Average of each feature per class
        self.means_ = np.zeros((n_classes, n_features))
        self.variances_ = np.zeros((n_classes, n_features))

        for idx, cls in enumerate(self.classes_):
            X_cls = X[y == cls]  # All samples belonging to this class
            
            # Mean across each feature
            self.means_[idx, :] = np.mean(X_cls, axis=0)
            
            # Variance across each feature 
            # Adding var_smoothing to avoid division by zero and for numerical stability
            self.variances_[idx, :] = np.var(X_cls, axis=0) + self.var_smoothing

        return self

    def _log_likelihood(self, X):
        """
        Calculate log of the Gaussian likelihood: log P(X | y)
        This is the core "generative" engine of Naive Bayes.
        """
        n_samples, n_features = X.shape
        n_classes = len(self.classes_)
        
        # We will store log likelihood for each sample per class
        # Shape: (n_samples, n_classes)
        log_likelihood = np.zeros((n_samples, n_classes))
        
        for idx in range(n_classes):
            # Extract parameters for this class
            mean = self.means_[idx, :]
            var = self.variances_[idx, :]
            
            # --- The Gaussian PDF (log scale) ---
            # 1. Constant term: -0.5 * log(2 * pi * variance)
            # 2. Quadratic term: -0.5 * ( (X - mean)^2 / variance )
            # Vectorized over all samples and all features.
            
            # Constant term (applies to all features and samples)
            log_const = -0.5 * np.sum(np.log(2 * np.pi * var))
            
            # Quadratic term: sum over features
            # (X - mean)^2 / var
            # We use keepdims=False to sum across features (axis=1)
            exponent = -0.5 * np.sum(((X - mean) ** 2) / var, axis=1)
            
            # Total log likelihood for this class
            log_likelihood[:, idx] = log_const + exponent
            
        return log_likelihood

    def predict_log_proba(self, X):
        """
        Calculate log probabilities P(y | X).
        Uses Log-Sum-Exp trick for numerical stability.
        """
        # 1. Log prior: log(P(y))
        log_prior = np.log(self.priors_)
        
        # 2. Log likelihood: log(P(X | y))
        log_likelihood = self._log_likelihood(X)
        
        # 3. Joint log probability: log(P(y)) + log(P(X | y))
        log_joint = log_likelihood + log_prior  # Shape: (n_samples, n_classes)
        
        # 4. Normalize using Log-Sum-Exp trick to prevent overflow
        # We want log( exp(log_joint) / sum(exp(log_joint)) )
        # = log_joint - log(sum(exp(log_joint)))
        log_sum_exp = np.log(np.sum(np.exp(log_joint), axis=1, keepdims=True))
        log_proba = log_joint - log_sum_exp
        
        return log_proba

    def predict_proba(self, X):
        """Return probability estimates for each class (exponentiate log)."""
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        """Return the predicted class with the highest probability."""
        log_proba = self.predict_log_proba(X)
        return self.classes_[np.argmax(log_proba, axis=1)]